# Agent lab : un agent sur les lignes Dakar Dem Dikk

Atelier Claude Code, épisode E11, adapté à UrbanFlow.

L'agent répond aux questions sur les lignes DDD à partir d'un seul fichier : `urbanflow_lignes_complet.csv` (42 lignes, données du projet UrbanFlow). Il appelle deux outils Python pour chercher dans ce fichier. Quand une donnée manque, il répond « non relevé ».

1. **Sans mémoire** : chaque question part seule. La question de suivi échoue.
2. **Avec mémoire** : une session de discussion garde l'historique. La question de suivi réussit.

Modèle : Gemini (SDK officiel `google-genai`), clé lue dans `.env`. Lance les cellules dans l'ordre avec le noyau **agents-lab**.

## 1. Préparation
On charge la clé depuis `.env` (jamais affichée) et le fichier des lignes.

In [ ]:
import os
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from google import genai
from google.genai import types

load_dotenv(find_dotenv(usecwd=True))  # lit GEMINI_API_KEY dans .env (dossier courant ou parent), sans l'afficher
assert os.getenv("GEMINI_API_KEY"), "Colle ta clé dans .env (GEMINI_API_KEY=...), puis relance cette cellule."
MODELE = os.getenv("GEMINI_MODEL") or "gemini-3.5-flash-lite"
client = genai.Client()  # prend GEMINI_API_KEY dans l'environnement

# Le notebook peut être lancé depuis 11-agent/ ou depuis notebooks/
CSV = next(p for p in [Path("../urbanflow-app/dify/urbanflow_lignes_complet.csv"),
                       Path("../../urbanflow-app/dify/urbanflow_lignes_complet.csv")] if p.exists())
lignes = pd.read_csv(CSV, encoding="utf-8-sig", dtype=str).fillna("")
print(len(lignes), "lignes chargées depuis", CSV.name)
lignes[["ligne", "categorie", "trajet_officiel"]].head()

## 2. Les outils de l'agent
Deux fonctions Python que le modèle peut appeler. Elles ne renvoient que ce qui est dans le fichier ; un champ vide devient « non relevé ». Le SDK lit leur nom, leur description et leurs paramètres pour savoir quand les appeler.

In [ ]:
def _fiche(r) -> dict:
    v = lambda c: r[c].strip() or "non relevé"
    return {
        "ligne": v("ligne"), "categorie": v("categorie"), "trajet": v("trajet_officiel"),
        "terminus": v("terminus_captures_appli"), "zones_traversees": v("zones_traversees"),
        "heures_de_pointe": v("heures_pointe_habituelles"),
        "a_verifier": r["a_verifier"].strip() or "non",
    }

def chercher_ligne(numero: str) -> dict:
    """Donne la fiche d'une ligne Dakar Dem Dikk à partir de son numéro (ex : "7", "18", "TAF TAF 1")."""
    n = numero.strip().lower().removeprefix("ligne").strip()
    trouve = lignes[lignes["ligne"].str.lower() == n]
    return _fiche(trouve.iloc[0]) if len(trouve) else {"erreur": f"ligne {n.upper()} non relevée dans le fichier"}

def lignes_desservant(lieu: str) -> list[dict]:
    """Liste les lignes dont le trajet, les terminus ou les zones traversées contiennent ce lieu (ex : "Ouakam", "Plateau")."""
    l = lieu.strip().lower()
    cols = ["trajet_officiel", "terminus_captures_appli", "zones_traversees"]
    masque = lignes[cols].apply(lambda c: c.str.lower().str.contains(l, regex=False)).any(axis=1)
    return [_fiche(r) for _, r in lignes[masque].head(8).iterrows()] or [{"erreur": f"aucune ligne relevée pour {lieu}"}]

print(chercher_ligne("7")["trajet"])
print([f["ligne"] for f in lignes_desservant("Ouakam")])

## 3. Les consignes
Les mêmes règles que l'app UrbanFlow : aucune donnée inventée, ni temps de trajet en bus, ni affluence, ni horaire.

In [ ]:
CONSIGNES = """Tu es l'assistant UrbanFlow pour les bus Dakar Dem Dikk (Dakar). Réponds en français, en 3 phrases maximum.
Utilise uniquement les outils chercher_ligne et lignes_desservant. N'invente rien.
Si une information n'est pas dans les résultats des outils, écris « non relevé ».
Ne donne jamais de temps de trajet en bus, d'horaire, de prix ni d'affluence : ces données ne sont pas publiées."""

config = types.GenerateContentConfig(
    system_instruction=CONSIGNES,
    tools=[chercher_ligne, lignes_desservant],  # appel automatique des fonctions par le SDK
    temperature=0.2,
)
Q1 = "Quelle ligne va de Ouakam au Palais 2 ?"
Q2 = "Et quels quartiers traverse-t-elle, et combien de temps faut-il en bus ?"

## 4. Partie 1 : agent sans mémoire
Chaque appel est indépendant (appels asynchrones avec `client.aio`). À la deuxième question, l'agent ne sait plus de quelle ligne on parle.

Le SDK peut afficher un avertissement qui recommande les sessions de discussion pour l'appel automatique des outils : c'est justement ce qu'on montre en partie 2, on peut l'ignorer ici.

In [ ]:
async def demander_sans_memoire(question: str) -> str:
    r = await client.aio.models.generate_content(model=MODELE, contents=question, config=config)
    return r.text

print("Q1 :", Q1, "\n->", await demander_sans_memoire(Q1), "\n")
print("Q2 :", Q2, "\n->", await demander_sans_memoire(Q2))

**À observer** : la réponse à Q2 demande de préciser la ligne, ou cherche au hasard. Rien ne relie Q2 à Q1.

## 5. Partie 2 : agent avec mémoire de session
Une session de discussion (`client.aio.chats`) renvoie tout l'historique au modèle à chaque message. L'agent sait que « elle » désigne la ligne trouvée à Q1.

In [ ]:
session = client.aio.chats.create(model=MODELE, config=config)

r1 = await session.send_message(Q1)
print("Q1 :", Q1, "\n->", r1.text, "\n")
r2 = await session.send_message(Q2)
print("Q2 :", Q2, "\n->", r2.text)

**À observer** : Q2 donne les zones de la ligne 7 et dit que le temps en bus n'est pas relevé (il n'est pas publié par DDD).

Le nombre de messages gardés en mémoire :

In [ ]:
print(len(session.get_history()), "messages dans l'historique de la session")

## 6. Ce qu'il faut retenir
- **Sans mémoire**, chaque question coûte peu mais l'agent ne suit pas une conversation.
- **Avec mémoire**, l'historique est renvoyé à chaque message : l'agent suit la conversation, mais chaque message coûte plus de jetons.
- Les outils limitent l'agent aux données du fichier : il répond « non relevé » au lieu d'inventer.
- La clé reste dans `.env` (ignoré par git) et n'est jamais affichée.